In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers
!pip install torch torchvision torchaudio transformers


Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: transformers 4.49.0
Uninstalling transformers-4.49.0:
  Successfully uninstalled transformers-4.49.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.0

In [ ]:
from google.colab import drive
import os

#  Mount Google Drive
drive.mount('/content/drive',force_remount=True)

#  Set the dataset path
DATA_PATH = "/content/drive/MyDrive/MELD-Extracted"

#  Verify dataset availability
if os.path.exists(DATA_PATH):
    print("Dataset found! Listing files:")
    !ls -lh "{DATA_PATH}"
else:
    print("Dataset not found! Please check Google Drive sharing settings.")

Mounted at /content/drive
Dataset found! Listing files:
lrw------- 1 root root 0 Feb 20 04:18 /content/drive/MyDrive/MELD-Extracted -> /content/drive/.shortcut-targets-by-id/1dg1QQKMtMmbXwmWPahUjjCNVfxD_O31h/MELD-Extracted


In [ ]:
import pandas as pd
import numpy as np
# 使用正确的文件路径读取文件
file_path = "/content/drive/MyDrive/MELD-Extracted/train/train_sent_emo.csv"
file_path_test = "/content/drive/MyDrive/MELD-Extracted/test/test_sent_emo.csv"
data = pd.read_csv(file_path)
data_test = pd.read_csv(file_path_test)
data_val = pd.read_csv("/content/drive/MyDrive/MELD-Extracted/dev/dev_sent_emo.csv")
# Train
# texts = data['Utterance']
# labels = data['Emotion']
# val_dataset = test_data

In [ ]:
print(data.shape)
print(data_test.shape)
print(data_val.shape)
print(data['Emotion'].value_counts())


(9989, 11)
(2610, 11)
(1109, 11)
Emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64


In [ ]:

def get_context(df, max_sequence=5):
    df = df.sort_values(by='Dialogue_ID')  # 确保按对话顺序排序
    df['context'] = df['Utterance'].astype(str)  # 初始化 context 列

    # 按 Dialogue_ID 分组
    grouped = df.groupby('Dialogue_ID')

    new_contexts = []

    for _, group in grouped:
        if len(group) <= max_sequence:
            # 若对话长度 ≤ max_sequence，将所有 Utterance 连接
            context_str = ' '.join(group['Utterance'])
            df.loc[group.index, 'context'] = context_str
        else:
            # 按原逻辑添加上下文
            for i in range(1, 3):
                df.loc[group.index, 'context'] = df.loc[group.index, 'context'].str.cat(
                    df.loc[group.index, 'Utterance'].shift(i, fill_value=''), sep=' '
                )
                df.loc[group.index, 'context'] = df.loc[group.index, 'context'].str.cat(
                    df.loc[group.index, 'Utterance'].shift(-i, fill_value=''), sep=' '
                )

    return df

train_data = data.groupby('Dialogue_ID', group_keys=False).apply(get_context)
test_data = data_test.groupby('Dialogue_ID', group_keys=False).apply(get_context)
val_data = data_val.groupby('Dialogue_ID', group_keys=False).apply(get_context)



<ipython-input-4-3019547c2625>:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data = data.groupby('Dialogue_ID', group_keys=False).apply(get_context)
<ipython-input-4-3019547c2625>:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_data = data_test.groupby('Dialogue_ID', group_keys=False).apply(get_context)
<ipython-input-4-3019547c2625>:29: DeprecationWarning: DataFrameGroupBy.apply operated o

In [ ]:
print(val_data['context'].head(5))
print(train_data.shape)
print(test_data.shape)
print(val_data.shape)


0    Oh my God, hes lost it. Hes totally lost it....
1    Oh my God, hes lost it. Hes totally lost it....
2    Or! Or, we could go to the bank, close our acc...
3    Youre a genius! Or! Or, we could go to the ba...
4    Aww, man, now we wont be bank buddies! Youre...
Name: context, dtype: object
(9989, 12)
(2610, 12)
(1109, 12)


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking
from sklearn.preprocessing import LabelEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 加载 BERT 模型和 Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)  # 送到 GPU
bert_model.eval()  # 进入评估模式，关闭 dropou

# 让 BERT 在 GPU 上计算 embedding
def get_bert_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", padding='max_length', truncation=True, max_length=128).to(device)  # 送入 GPU
    with torch.no_grad():
        output = bert_model(**tokens)
    return output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()  # 计算完后转回 CPU

# 计算训练集、测试集和验证集 embedding
train_data['embedding'] = train_data['context'].apply(get_bert_embedding)
test_data['embedding'] = test_data['context'].apply(get_bert_embedding)
val_data['embedding'] = val_data['context'].apply(get_bert_embedding)


Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
print(type(train_data["embedding"]))
print(type(train_data["embedding"].iloc[0]))
print(train_data["embedding"].shape)  # (9989,)


<class 'pandas.core.series.Series'>
<class 'numpy.ndarray'>
(9989,)


In [ ]:
import numpy as np

def group_conversations(df, max_seq_length=5):
    grouped = []
    labels = []

    for dialogue_id in df['Dialogue_ID'].unique():
        conversation = df[df['Dialogue_ID'] == dialogue_id].sort_values(by="Utterance_ID")

        embeddings = np.vstack(conversation['embedding'].values)  # (num_utterances, 768)
        emotions = conversation['Emotion'].values  # (num_utterances,)

        # 处理不同长度的对话
        if embeddings.shape[0] < max_seq_length:
            pad = np.zeros((max_seq_length - embeddings.shape[0], 768))  # 补零
            embeddings = np.vstack([embeddings, pad])
            emotions = np.pad(emotions, (0, max_seq_length - len(emotions)), mode='constant', constant_values=-1)
        else:
            embeddings = embeddings[:max_seq_length]
            emotions = emotions[:max_seq_length]

        grouped.append(embeddings)
        labels.append(emotions)

    return np.array(grouped), np.array(labels)
X_train, y_train = group_conversations(train_data)
X_val, y_val = group_conversations(val_data)
X_test, y_test = group_conversations(test_data)



In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)
print(X_test.shape)
print(y_test.shape)


(1038, 5, 768)
(1038, 5)
(114, 5, 768)
(114, 5)
(280, 5, 768)
(280, 5)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = np.array(y_train, dtype=str)
y_val = np.array(y_val, dtype=str)
y_test = np.array(y_test, dtype=str)

y_train_flat = y_train.flatten()
y_val_flat = y_val.flatten()
y_test_flat = y_test.flatten()

label_encoder.fit(np.concatenate([y_train_flat, y_val_flat, y_test_flat]))

y_train_encoded = label_encoder.transform(y_train_flat).reshape(y_train.shape)
y_val_encoded = label_encoder.transform(y_val_flat).reshape(y_val.shape)
y_test_encoded = label_encoder.transform(y_test_flat).reshape(y_test.shape)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout, TimeDistributed, Masking
from tensorflow.keras.optimizers import Adam
model = Sequential([
    Masking(mask_value=0.0, input_shape=(5, 768)),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(len(label_encoder.classes_), activation='softmax')
])

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)


model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train_encoded, epochs=20, batch_size=64, validation_data=(X_val, y_val_encoded), callbacks=[reduce_lr, early_stopping])

Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 19s 182ms/step - accuracy: 0.3596 - loss: 1.7903 - val_accuracy: 0.4281 - val_loss: 1.5734 - learning_rate: 0.0010
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.4127 - loss: 1.5907 - val_accuracy: 0.4281 - val_loss: 1.5018 - learning_rate: 0.0010
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.4265 - loss: 1.5033 - val_accuracy: 0.4368 - val_loss: 1.4658 - learning_rate: 0.0010
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.4377 - loss: 1.4527 - val_accuracy: 0.4368 - val_loss: 1.4436 - learning_rate: 0.0010
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.4366 - loss: 1.4464 - val_accuracy: 0.4404 - val_loss: 1.4019 - learning_rate: 0.0010
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.4291 - loss: 1.4287 - val_accuracy: 0.4263 - val_loss: 1.4419 - learning_rate: 0.0010
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.4394 - loss: 1.4163 - val_a

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_encoded)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4703 - loss: 1.3215
Test Loss: 1.3152
Test Accuracy: 0.4829


In [ ]:
# 总参数
total_params = model.count_params()
print(f"Total Parameters: {total_params:,}")


Total Parameters: 1,354,440


# Trash


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking, Bidirectional
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf


y_train = np.array(y_train, dtype=str)
y_test = np.array(y_test, dtype=str)

y_train_flat = y_train.flatten()  # 变成 1D
y_test_flat = y_test.flatten()

label_encoder = LabelEncoder()
label_encoder.fit(np.concatenate([y_train_flat, y_test_flat]))  # 确保所有标签都见过

y_train_encoded = label_encoder.transform(y_train_flat).reshape(y_train.shape)  # 转回 2D
y_test_encoded = label_encoder.transform(y_test_flat).reshape(y_test.shape)


model = Sequential([
    Masking(mask_value=0.0, input_shape=(5, 768)),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.15, recurrent_dropout=0.15)),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.15)),
    Dense(len(label_encoder.classes_), activation='softmax')
])


initial_learning_rate = 0.0005
decay_steps = 50

decay_rate = 0.80

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=decay_steps,
    decay_rate=decay_rate,
    staircase=False  # 若为 True，则学习率以阶梯状变化
)

# 在优化器中使用该学习率调度
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

sample_weights = np.where(y_train == '-1', 0, 1)  # padding 权重设为 0，其余为 1
model.fit(X_train, y_train_encoded, epochs = 20, batch_size=32, validation_data=(X_test, y_test_encoded), sample_weight=sample_weights)



# 评估模型
test_loss, test_acc = model.evaluate(X_test, y_test_encoded)
print("Test Accuracy:", test_acc)


Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 16s 93ms/step - accuracy: 0.1536 - loss: 2.0332 - val_accuracy: 0.2136 - val_loss: 1.9062
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0965 - loss: 1.9523 - val_accuracy: 0.1643 - val_loss: 1.9566
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1131 - loss: 1.9108 - val_accuracy: 0.2207 - val_loss: 1.9190
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1402 - loss: 1.8580 - val_accuracy: 0.1764 - val_loss: 1.9595
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1533 - loss: 1.8227 - val_accuracy: 0.2543 - val_loss: 1.8714
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1519 - loss: 1.7758 - val_accuracy: 0.2743 - val_loss: 1.8186
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.1613 - loss: 1.7356 - val_accuracy: 0.2714 - val_loss: 1.8318
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.1810 - loss: 1.6937 - val_accuracy: 0.2150 - val_loss: 1

Only focus on emotion which has enough data

# Only focus on emotion which has enough data

Only focus on emotion which has enough data

In [ ]:
selected_emotions = ["neutral", "joy", "surprise", "anger"]

# 过滤数据集
filtered_data = data[data['Emotion'].isin(selected_emotions)]
print(filtered_data['Emotion'].value_counts())

Emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
Name: count, dtype: int64


In [ ]:
train_data = filtered_data.groupby('Dialogue_ID', group_keys=False).apply(get_context)
test_data = test_data.groupby('Dialogue_ID', group_keys=False).apply(get_context)

<ipython-input-56-d33fec43af82>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data = filtered_data.groupby('Dialogue_ID', group_keys=False).apply(get_context)
<ipython-input-56-d33fec43af82>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_data = test_data.groupby('Dialogue_ID', group_keys=False).apply(get_context)


In [ ]:
train_data['embedding'] = train_data['context'].apply(get_bert_embedding)
test_data['embedding'] = test_data['context'].apply(get_bert_embedding)

In [ ]:
print(train_data['embedding'].shape)
print(train_data.shape)

(8767,)
(8767, 13)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking, Bidirectional
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf


y_train = np.array(y_train, dtype=str)
y_test = np.array(y_test, dtype=str)

y_train_flat = y_train.flatten()  # 变成 1D
y_test_flat = y_test.flatten()

label_encoder = LabelEncoder()
label_encoder.fit(np.concatenate([y_train_flat, y_test_flat]))  # 确保所有标签都见过

y_train_encoded = label_encoder.transform(y_train_flat).reshape(y_train.shape)  # 转回 2D
y_test_encoded = label_encoder.transform(y_test_flat).reshape(y_test.shape)


model = Sequential([
    Masking(mask_value=0.0, input_shape=(5, 768)),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.15, recurrent_dropout=0.15)),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.15)),
    Dense(len(label_encoder.classes_), activation='softmax')
])


initial_learning_rate = 0.005
decay_steps = 50

decay_rate = 0.80

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=decay_steps,
    decay_rate=decay_rate,
    staircase=False  # 若为 True，则学习率以阶梯状变化
)

# 在优化器中使用该学习率调度
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

sample_weights = np.where(y_train == '-1', 0, 1)  # padding 权重设为 0，其余为 1
model.fit(X_train, y_train_encoded, epochs = 20, batch_size=32, validation_data=(X_test, y_test_encoded), sample_weight=sample_weights)



# 评估模型
test_loss, test_acc = model.evaluate(X_test, y_test_encoded)
print("Test Accuracy:", test_acc)


Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 14s 98ms/step - accuracy: 0.1536 - loss: 2.0021 - val_accuracy: 0.0757 - val_loss: 2.0256
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1145 - loss: 1.8738 - val_accuracy: 0.0964 - val_loss: 2.0209
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1447 - loss: 1.7946 - val_accuracy: 0.2979 - val_loss: 1.8336
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1631 - loss: 1.7040 - val_accuracy: 0.2729 - val_loss: 1.7538
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1674 - loss: 1.6901 - val_accuracy: 0.1593 - val_loss: 1.9568
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1654 - loss: 1.6689 - val_accuracy: 0.2093 - val_loss: 1.8551
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1817 - loss: 1.5819 - val_accuracy: 0.2493 - val_loss: 1.9429
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1991 - loss: 1.5294 - val_accuracy: 0.2143 - val_loss: 1